<a href="https://colab.research.google.com/github/alpacaYiChun/ML/blob/master/Alimama_hard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# FULL RUNNABLE COLAB CELL - Taobao Ad Sequential Recall
#
# Fixed / improved:
# 1) fixed NameError: best_active -> best_active_mask
# 2) removed future-click leakage in negative sampling
# 3) expanded user coverage:
#    - top users keep explicit ids
#    - long-tail users go to hashed OOV buckets
# 4) pid is now actually used:
#    - target pid in user/context tower
#    - history pid in sequence events
# 5) softer hard negatives:
#    - HARD_NEG_FRAC = 0.10
#    - OHEM only from exposed-not-clicked candidate pool
#    - random refill used only for easy negatives
# 6) fixed tau schedule: warmup to TAU_END, then stay there
# 7) masked in-batch loss for duplicate positive items
# 8) early stopping / best checkpoint based on recall@50
# ============================================================

!pip -q install -U "tqdm>=4.67"

import os
import tarfile
import random
import math
import copy
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)

# -----------------------
# Paths + download
# -----------------------
DATA_DIR = "/content/taobao_ad"
os.makedirs(DATA_DIR, exist_ok=True)

files = {
    "ad_feature.csv.tar":   "https://zenodo.org/records/8088629/files/ad_feature.csv.tar?download=1",
    "raw_sample.csv.tar":   "https://zenodo.org/records/8088629/files/raw_sample.csv.tar?download=1",
    "user_profile.csv.tar": "https://zenodo.org/records/8088629/files/user_profile.csv.tar?download=1",
}

def extract_tar(path):
    with tarfile.open(path, "r") as tar:
        tar.extractall(os.path.dirname(path))

for fname, url in files.items():
    out_path = os.path.join(DATA_DIR, fname)
    if not os.path.exists(out_path):
        print("Downloading", fname)
        !wget -q --show-progress -O "$out_path" "$url"
    extract_tar(out_path)

AD_PATH  = os.path.join(DATA_DIR, "ad_feature.csv")
RAW_PATH = os.path.join(DATA_DIR, "raw_sample.csv")
UP_PATH  = os.path.join(DATA_DIR, "user_profile.csv")

for p in [AD_PATH, RAW_PATH, UP_PATH]:
    if not os.path.exists(p):
        raise FileNotFoundError(p)

print("Files ready")

# -----------------------
# Config
# -----------------------
MAX_RAW_ROWS = None

TARGET_CLICK_COVERAGE_USER = 0.95
TARGET_CLICK_COVERAGE_ITEM = 0.95

TOPK_CAP_USER = 400_000
USER_HASH_BUCKETS = 200_000
TOPK_CAP_ITEM = 200_000

RECENT_DAYS = 14
TARGET_POOL_COVERAGE_VAL  = 0.95
TARGET_POOL_COVERAGE_TEST = 0.95

MAX_HIST_CLICKS = 50
SHORT_HIST_CLICKS = 10
LONG_HIST_MAX = 40

BATCH_SIZE = 768
NUM_WORKERS = 2

SEQ_HIDDEN = 256
MLP_HIDDEN = 512
DROPOUT = 0.1

HISTORY_ENCODER = "gru_transformer"   # "gru" | "transformer" | "gru_transformer"

TRANSFORMER_NHEAD = 8
TRANSFORMER_FF_DIM = 1024
TRANSFORMER_LAYERS = 2
TRANSFORMER_DROPOUT = 0.1

K_NEG = 255
HARD_NEG_FRAC = 0.10
HARD_CAND_MULT = 4

HARD_NEG_KEEP = max(1, int(round(K_NEG * HARD_NEG_FRAC)))
EASY_NEG_KEEP = K_NEG - HARD_NEG_KEEP
HARD_NEG_CAND = max(HARD_NEG_KEEP * HARD_CAND_MULT, HARD_NEG_KEEP)

LOSS_A_MULT = 0.5   # masked in-batch InfoNCE
LOSS_B_MULT = 3.0   # sampled InfoNCE

LR = 2e-4
WEIGHT_DECAY = 1e-5
GRAD_CLIP = 5.0

EPOCHS = 20
TAU_START = 0.20
TAU_END   = 0.07
TAU_WARMUP_EPOCHS = 5
TAU_VAL = 0.07

TIME_GAP_BUCKETS = 32

MONITOR_METRIC = "recall@50"
EARLY_STOP_PATIENCE = 4
LR_SCHED_PATIENCE = 1
LR_SCHED_FACTOR = 0.5
MIN_LR = 1e-6

print({
    "TOPK_CAP_USER": TOPK_CAP_USER,
    "USER_HASH_BUCKETS": USER_HASH_BUCKETS,
    "K_NEG": K_NEG,
    "EASY_NEG_KEEP": EASY_NEG_KEEP,
    "HARD_NEG_KEEP": HARD_NEG_KEEP,
    "HARD_NEG_CAND": HARD_NEG_CAND,
    "LOSS_A_MULT": LOSS_A_MULT,
    "LOSS_B_MULT": LOSS_B_MULT,
    "BATCH_SIZE": BATCH_SIZE,
    "MONITOR_METRIC": MONITOR_METRIC,
})

# -----------------------
# Load data
# -----------------------
ad = pd.read_csv(AD_PATH, low_memory=False)
up = pd.read_csv(UP_PATH, low_memory=False)

raw = pd.read_csv(
    RAW_PATH,
    nrows=MAX_RAW_ROWS,
    header=None,
    low_memory=False,
    dtype="string"
)

raw.columns = raw.iloc[0]
raw = raw.iloc[1:].reset_index(drop=True)

# -----------------------
# Clean + types
# -----------------------
ad["adgroup_id"] = pd.to_numeric(ad["adgroup_id"], errors="coerce").astype("Int64")
ad["cate_id"]    = pd.to_numeric(ad["cate_id"], errors="coerce").astype("Int64")
ad["brand"]      = pd.to_numeric(ad["brand"], errors="coerce").astype("Int64")
ad["price"]      = pd.to_numeric(ad["price"], errors="coerce")

up = up.drop(columns=["pvalue_level", "new_user_class_level"], errors="ignore")
up["userid"] = pd.to_numeric(up["userid"], errors="coerce").astype("Int64")

for c in ["cms_segid", "cms_group_id", "final_gender_code", "age_level", "shopping_level", "occupation"]:
    up[c] = pd.to_numeric(up[c], errors="coerce").fillna(0).astype(np.int64)

raw["user"]       = pd.to_numeric(raw["user"], errors="coerce").astype("Int64")
raw["adgroup_id"] = pd.to_numeric(raw["adgroup_id"], errors="coerce").astype("Int64")
raw["clk"]        = pd.to_numeric(raw["clk"], errors="coerce").fillna(0).astype(np.int64)
raw["nonclk"]     = pd.to_numeric(raw["nonclk"], errors="coerce").fillna(0).astype(np.int64)
raw["time_stamp"] = pd.to_numeric(raw["time_stamp"], errors="coerce").fillna(0).astype(np.int64)
raw["pid"]        = raw["pid"].astype("string")

dt = pd.to_datetime(raw["time_stamp"], unit="s", utc=True, errors="coerce")
raw["month"] = dt.dt.month.fillna(1).astype(np.int64)

raw = raw[raw["user"].notna() & raw["adgroup_id"].notna()].reset_index(drop=True)
print("raw rows:", len(raw))

# -----------------------
# Helper: choose TopK by click coverage
# -----------------------
def topk_by_click_coverage(ids: np.ndarray, is_click: np.ndarray, target_cov: float, cap: int):
    m = is_click.astype(np.int64) == 1
    ids_click = ids[m]

    vc = pd.Series(ids_click).value_counts().sort_values(ascending=False)
    counts = vc.to_numpy(dtype=np.int64)

    total = counts.sum()
    if total == 0:
        return {}, 0, 0.0

    cumsum = np.cumsum(counts)
    k = int(np.searchsorted(cumsum, target_cov * total) + 1)
    k = min(k, cap, len(vc))

    keep = vc.head(k).index.to_numpy()
    id2idx = {int(kid): i + 1 for i, kid in enumerate(keep)}

    achieved = float(cumsum[k - 1] / total) if k > 0 else 0.0
    return id2idx, k, achieved

u_raw_vals = raw["user"].astype(np.int64).values
i_raw_vals = raw["adgroup_id"].astype(np.int64).values
clk_vals   = raw["clk"].astype(np.int64).values

user_top_id2idx, USER_TOPK, user_cov = topk_by_click_coverage(
    u_raw_vals,
    clk_vals,
    TARGET_CLICK_COVERAGE_USER,
    TOPK_CAP_USER
)

adg_id2idx, ADG_TOPK, item_cov = topk_by_click_coverage(
    i_raw_vals,
    clk_vals,
    TARGET_CLICK_COVERAGE_ITEM,
    TOPK_CAP_ITEM
)

print(f"USER_TOPK chosen={USER_TOPK} covers click volume={user_cov:.4f} target={TARGET_CLICK_COVERAGE_USER}")
print(f"ADG_TOPK  chosen={ADG_TOPK} covers click volume={item_cov:.4f} target={TARGET_CLICK_COVERAGE_ITEM}")

def stable_hash_bucket_int(x: int, num_buckets: int) -> int:
    if num_buckets <= 0:
        return 0
    x = int(x) & 0xFFFFFFFF
    return ((x * 2654435761) & 0xFFFFFFFF) % num_buckets

def map_user_code(u_raw: int) -> int:
    """
    1..USER_TOPK for top users
    USER_TOPK+1 .. USER_TOPK+USER_HASH_BUCKETS for long-tail hashed buckets
    0 reserved for padding only
    """
    code = user_top_id2idx.get(int(u_raw), None)
    if code is not None:
        return code
    if USER_HASH_BUCKETS > 0:
        return USER_TOPK + 1 + stable_hash_bucket_int(int(u_raw), USER_HASH_BUCKETS)
    return 0

cate_unique = pd.Index(ad["cate_id"].dropna().astype(np.int64).unique())
cate_id2idx = {int(k): i + 1 for i, k in enumerate(cate_unique)}

def topk_id2idx(values: np.ndarray, topk: int):
    vc = pd.Series(values).value_counts()
    keep = vc.head(topk).index.to_numpy()
    return {int(k): i + 1 for i, k in enumerate(keep)}

BRAND_TOPK = 2000
brand_vals = ad["brand"].dropna().astype(np.int64).values
brand_id2idx = topk_id2idx(brand_vals, BRAND_TOPK)

pid_unique = pd.Index(raw["pid"].dropna().unique())
pid2idx = {str(k): i + 1 for i, k in enumerate(pid_unique)}

USER_V  = USER_TOPK + USER_HASH_BUCKETS + 1
ADG_V   = len(adg_id2idx) + 1
CATE_V  = len(cate_id2idx) + 1
BRAND_V = len(brand_id2idx) + 1
PID_V   = len(pid2idx) + 1

MONTH_V = 13
TIME_GAP_V = TIME_GAP_BUCKETS + 1

CMS_SEG_V = int(up["cms_segid"].max()) + 2
CMS_GRP_V = int(up["cms_group_id"].max()) + 2
GENDER_V  = int(up["final_gender_code"].max()) + 2
AGE_V     = int(up["age_level"].max()) + 2
SHOP_V    = int(up["shopping_level"].max()) + 2
OCC_V     = int(up["occupation"].max()) + 2

print("Vocabs:", {
    "USER_V": USER_V,
    "ADG_V": ADG_V,
    "CATE_V": CATE_V,
    "BRAND_V": BRAND_V,
    "PID_V": PID_V,
})

# -----------------------
# Field-aware embedding dims
# -----------------------
def suggest_dim(cardinality, min_dim=8, max_dim=128, multiplier=1.6):
    if cardinality <= 2:
        return min_dim
    d = int(math.ceil(multiplier * (float(cardinality) ** 0.25)))
    d = int(math.ceil(d / 8.0) * 8)
    d = max(min_dim, min(max_dim, d))
    return d

D_USER   = suggest_dim(USER_V, 32, 96)
D_ADG    = suggest_dim(ADG_V, 64, 128)
D_CATE   = suggest_dim(CATE_V, 16, 48)
D_BRAND  = suggest_dim(BRAND_V, 16, 48)
D_PID    = suggest_dim(PID_V, 8, 24)

D_CMS_SEG = suggest_dim(CMS_SEG_V, 8, 24)
D_CMS_GRP = suggest_dim(CMS_GRP_V, 8, 24)
D_GENDER  = suggest_dim(GENDER_V, 8, 16)
D_AGE     = suggest_dim(AGE_V, 8, 16)
D_SHOP    = suggest_dim(SHOP_V, 8, 16)
D_OCC     = suggest_dim(OCC_V, 8, 16)

D_GAP   = 8
D_MONTH = 8

ITEM_OUT_DIM = 128

print("Embedding dims:", {
    "user": D_USER, "adg": D_ADG, "cate": D_CATE, "brand": D_BRAND, "pid": D_PID,
    "cms_seg": D_CMS_SEG, "cms_grp": D_CMS_GRP, "gender": D_GENDER,
    "age": D_AGE, "shop": D_SHOP, "occ": D_OCC, "gap": D_GAP, "month": D_MONTH
})

# -----------------------
# Price normalization: log1p + per-category robust stats
# -----------------------
ad["price"] = pd.to_numeric(ad["price"], errors="coerce")
ad["price"] = ad["price"].clip(lower=0)
ad["log_price"] = np.log1p(ad["price"].fillna(0.0))

global_med = float(ad["log_price"].median())
global_q25 = float(ad["log_price"].quantile(0.25))
global_q75 = float(ad["log_price"].quantile(0.75))
global_iqr = max(global_q75 - global_q25, 1e-6)

tmp_price = ad[["cate_id", "log_price"]].copy()
cate_stats = (
    tmp_price
    .dropna(subset=["cate_id"])
    .groupby("cate_id")["log_price"]
    .agg(["count", "median"])
    .rename(columns={"count": "cnt", "median": "med"})
)

cate_q25 = tmp_price.dropna(subset=["cate_id"]).groupby("cate_id")["log_price"].quantile(0.25).rename("q25")
cate_q75 = tmp_price.dropna(subset=["cate_id"]).groupby("cate_id")["log_price"].quantile(0.75).rename("q75")
cate_stats = cate_stats.join(cate_q25).join(cate_q75)
cate_stats["iqr"] = (cate_stats["q75"] - cate_stats["q25"]).clip(lower=1e-6)

def normalize_price_by_category(row):
    lp = float(row["log_price"]) if pd.notna(row["log_price"]) else 0.0
    cid = row["cate_id"]
    if pd.isna(cid) or cid not in cate_stats.index or cate_stats.loc[cid, "cnt"] < 20:
        z = (lp - global_med) / global_iqr
    else:
        med = float(cate_stats.loc[cid, "med"])
        iqr = float(cate_stats.loc[cid, "iqr"])
        z = (lp - med) / iqr
    return float(np.clip(z, -5.0, 5.0))

ad["price_norm"] = ad.apply(normalize_price_by_category, axis=1).astype(np.float32)

# -----------------------
# Item feature tables indexed by adg_code
# -----------------------
cate_by_adg  = np.zeros(ADG_V, dtype=np.int64)
brand_by_adg = np.zeros(ADG_V, dtype=np.int64)
price_by_adg = np.zeros(ADG_V, dtype=np.float32)

ad_id = ad["adgroup_id"].astype("Int64").to_numpy()
cate_id = ad["cate_id"].astype("Int64").to_numpy()
brand_id = ad["brand"].astype("Int64").to_numpy()
price_norm_all = ad["price_norm"].to_numpy(dtype=np.float32)

for idx in range(len(ad)):
    aid = ad_id[idx]
    if pd.isna(aid):
        continue

    adg_code = adg_id2idx.get(int(aid), 0)
    if adg_code == 0:
        continue

    cid = cate_id[idx]
    bid = brand_id[idx]

    ccode = cate_id2idx.get(int(cid), 0) if not pd.isna(cid) else 0
    bcode = brand_id2idx.get(int(bid), 0) if not pd.isna(bid) else 0

    cate_by_adg[adg_code]  = ccode
    brand_by_adg[adg_code] = bcode
    price_by_adg[adg_code] = float(price_norm_all[idx])

# -----------------------
# Build per-user sorted sequences
# -----------------------
raw_pid_code = np.array([pid2idx.get(str(x), 0) for x in raw["pid"].values], dtype=np.int64)

tmp = pd.DataFrame({
    "user": raw["user"].astype(np.int64).values,
    "adg_raw": raw["adgroup_id"].astype(np.int64).values,
    "pid_code": raw_pid_code,
    "month": raw["month"].astype(np.int64).values,
    "clk": raw["clk"].astype(np.int64).values,
    "non": raw["nonclk"].astype(np.int64).values,
    "ts": raw["time_stamp"].astype(np.int64).values
}).sort_values(["user", "ts"]).reset_index(drop=True)

u_static = up.set_index(up["userid"].astype(np.int64))[[
    "cms_segid",
    "cms_group_id",
    "final_gender_code",
    "age_level",
    "shopping_level",
    "occupation"
]]

user_event = {}

for u, g in tqdm(tmp.groupby("user", sort=False), desc="building user_event"):
    adg_raw = g["adg_raw"].to_numpy(np.int64)
    pid_code = g["pid_code"].to_numpy(np.int64)
    mo  = g["month"].to_numpy(np.int64)
    clk = g["clk"].to_numpy(np.int64)
    non = g["non"].to_numpy(np.int64)
    ts  = g["ts"].to_numpy(np.int64)

    click_pos = np.where(clk == 1)[0]
    if len(click_pos) < 3:
        continue

    try:
        us = u_static.loc[int(u)]
        demo = np.array([
            int(us["cms_segid"]),
            int(us["cms_group_id"]),
            int(us["final_gender_code"]),
            int(us["age_level"]),
            int(us["shopping_level"]),
            int(us["occupation"]),
        ], dtype=np.int64)
    except KeyError:
        demo = np.zeros((6,), dtype=np.int64)

    user_event[int(u)] = {
        "demo": demo,
        "adg_raw": adg_raw,
        "pid_code": pid_code,
        "month": mo,
        "clk": clk,
        "non": non,
        "ts": ts,
        "click_pos": click_pos,
    }

print("Users kept:", len(user_event))
kept_in_top = sum(1 for u in user_event.keys() if int(u) in user_top_id2idx)
print(f"Kept users covered explicitly by top user vocab: {kept_in_top}/{len(user_event)} = {kept_in_top / max(len(user_event),1):.4f}")

# -----------------------
# Active/hot pool selection
# -----------------------
end_ts = int(raw["time_stamp"].max())
cut_ts = end_ts - int(RECENT_DAYS * 86400)

raw_adg_codes = np.array(
    [adg_id2idx.get(int(x), 0) for x in raw["adgroup_id"].astype(np.int64).values],
    dtype=np.int64
)

raw_clk = raw["clk"].astype(np.int64).values
raw_ts  = raw["time_stamp"].astype(np.int64).values

click_mask = (raw_clk == 1) & (raw_adg_codes != 0)
recent_mask = click_mask & (raw_ts >= cut_ts)

total_clicks = np.bincount(raw_adg_codes[click_mask], minlength=ADG_V).astype(np.int64)
recent_clicks = np.bincount(raw_adg_codes[recent_mask], minlength=ADG_V).astype(np.int64)

def split_label_coverage(user_event_dict, split, active_mask):
    total, covered = 0, 0

    for u, d in user_event_dict.items():
        cp = d["click_pos"]

        if split == "val":
            use = cp[-2:-1]
        elif split == "test":
            use = cp[-1:]
        else:
            continue

        for p in use:
            total += 1
            pos_raw = int(d["adg_raw"][int(p)])
            pos_code = adg_id2idx.get(pos_raw, 0)

            if pos_code != 0 and active_mask[pos_code]:
                covered += 1

    return covered / max(total, 1), covered, total

min_recent = 50
min_total  = 1

best_active_mask = None
best_stats = None

for trial in range(12):
    active_mask = (recent_clicks >= min_recent) & (total_clicks >= min_total)
    active_mask[0] = False

    val_cov, val_cov_n, val_tot = split_label_coverage(user_event, "val", active_mask)
    test_cov, test_cov_n, test_tot = split_label_coverage(user_event, "test", active_mask)
    pool_size = int(active_mask.sum())

    print(
        f"[POOL] min_recent={min_recent:>3} pool={pool_size:>6} "
        f"val_cov={val_cov:.3f} ({val_cov_n}/{val_tot}) "
        f"test_cov={test_cov:.3f} ({test_cov_n}/{test_tot})"
    )

    best_active_mask = active_mask
    best_stats = (min_recent, pool_size, val_cov, test_cov)

    if val_cov >= TARGET_POOL_COVERAGE_VAL and test_cov >= TARGET_POOL_COVERAGE_TEST:
        break

    min_recent = max(1, int(min_recent * 0.6))

active_mask = best_active_mask
ACTIVE_ADG_CODES = np.where(active_mask)[0].astype(np.int64)

print(
    f"Final active pool: size={len(ACTIVE_ADG_CODES)}, "
    f"min_recent={best_stats[0]}, "
    f"val_cov={best_stats[2]:.3f}, "
    f"test_cov={best_stats[3]:.3f}, "
    f"end_ts={end_ts}, cut_ts={cut_ts}"
)

# -----------------------
# Popularity sampler with proper truncation
# -----------------------
prob = np.zeros(ADG_V, dtype=np.float64)
active_counts = total_clicks[ACTIVE_ADG_CODES].astype(np.float64)
active_counts_pos = active_counts[active_counts > 0]

if len(active_counts_pos) > 0:
    pop_cap = float(np.quantile(active_counts_pos, 0.995))
else:
    pop_cap = 1.0

trunc_counts = np.minimum(total_clicks.astype(np.float64), pop_cap)
prob[ACTIVE_ADG_CODES] = np.power(np.maximum(trunc_counts[ACTIVE_ADG_CODES], 1.0), 0.75)
prob[0] = 0.0
prob = prob / (prob.sum() + 1e-12)

prob_t = torch.from_numpy(prob.astype(np.float32))
print(f"Popularity truncation cap={pop_cap:.3f}")

# -----------------------
# Time gap bucket
# -----------------------
def make_gap_bucket(target_ts: int, hist_ts: np.ndarray):
    delta_sec = np.maximum(target_ts - hist_ts, 0)
    delta_day = delta_sec.astype(np.float64) / 86400.0
    bucket = np.floor(np.log2(delta_day + 1.0)).astype(np.int64)
    bucket = np.clip(bucket, 0, TIME_GAP_BUCKETS - 1)
    return bucket

# -----------------------
# Dataset
# -----------------------
class ClickDatasetActive(Dataset):
    def __init__(self, user_event_dict, split: str, max_hist=50, deterministic=False, seed_base=1234):
        self.user_event = user_event_dict
        self.max_hist = max_hist
        self.deterministic = deterministic
        self.seed_base = int(seed_base)
        self.samples = []

        for u, d in self.user_event.items():
            cp = d["click_pos"]

            if split == "train":
                use = cp[:-2]
            elif split == "val":
                use = cp[-2:-1]
            elif split == "test":
                use = cp[-1:]
            else:
                raise ValueError(split)

            for p in use:
                pos_raw = int(d["adg_raw"][int(p)])
                pos_code = adg_id2idx.get(pos_raw, 0)

                if pos_code == 0 or not active_mask[pos_code]:
                    continue

                self.samples.append((u, int(p)))

    def __len__(self):
        return len(self.samples)

    def _get_generator(self, sample_idx):
        if not self.deterministic:
            return None
        g = torch.Generator(device="cpu")
        g.manual_seed(self.seed_base + int(sample_idx))
        return g

    def _draw_from_prob(self, num_samples, generator):
        if num_samples <= 0:
            return np.empty((0,), dtype=np.int64)
        cand = torch.multinomial(
            prob_t,
            num_samples=num_samples,
            replacement=True,
            generator=generator
        )
        return cand.numpy().astype(np.int64)

    def _random_refill(self, forbid_set, already_selected, need, generator):
        if need <= 0:
            return []

        forbid = set(forbid_set)
        forbid.update(already_selected)

        negs = []
        draw = max(need * 12, 128)

        tries = 0
        while len(negs) < need and tries < 8:
            cand = self._draw_from_prob(draw, generator)
            for c in cand:
                if c == 0:
                    continue
                if c in forbid:
                    continue
                if not active_mask[c]:
                    continue

                negs.append(int(c))
                forbid.add(int(c))
                if len(negs) == need:
                    break
            tries += 1

        if len(negs) < need:
            negs += [0] * (need - len(negs))

        return negs

    def _sample_negatives(self, d, click_pos, pos_adg, sample_idx):
        """
        Important fix:
        - only exclude items clicked BEFORE target click
        - do not exclude future clicks (no future leakage)
        """
        generator = self._get_generator(sample_idx)

        cp = d["click_pos"]
        idx = int(np.searchsorted(cp, click_pos))
        clicked_before_pos = cp[:idx]

        clicked_before_codes = np.array(
            [adg_id2idx.get(int(x), 0) for x in d["adg_raw"][clicked_before_pos].tolist()],
            dtype=np.int64
        )
        clicked_before_codes = np.unique(clicked_before_codes)
        clicked_before_codes = clicked_before_codes[clicked_before_codes != 0]

        forbid = set(clicked_before_codes.tolist())
        forbid.add(int(pos_adg))

        # exposures before target
        expo_pos = np.where(d["clk"][:click_pos] == 0)[0]
        expo_raw = d["adg_raw"][expo_pos].astype(np.int64)
        expo_codes = np.array([adg_id2idx.get(int(x), 0) for x in expo_raw], dtype=np.int64)
        expo_ts = d["ts"][expo_pos].astype(np.int64)

        expo_last_ts = {}
        for c, t in zip(expo_codes, expo_ts):
            if c == 0:
                continue
            if not active_mask[c]:
                continue
            if c in forbid:
                continue
            old = expo_last_ts.get(int(c), -1)
            if int(t) > old:
                expo_last_ts[int(c)] = int(t)

        expo_unique = list(expo_last_ts.keys())

        pos_cate = int(cate_by_adg[pos_adg]) if pos_adg else 0
        pos_brand = int(brand_by_adg[pos_adg]) if pos_adg else 0
        pos_price = float(price_by_adg[pos_adg]) if pos_adg else 0.0

        ranked = []
        for c in expo_unique:
            same_cate = int(cate_by_adg[c] == pos_cate and pos_cate != 0)
            same_brand = int(brand_by_adg[c] == pos_brand and pos_brand != 0)
            price_close = -abs(float(price_by_adg[c]) - pos_price)
            recency = expo_last_ts[c]
            ranked.append((same_cate, same_brand, price_close, recency, c))

        ranked.sort(reverse=True)

        # hard pool: only exposed-not-clicked, no random filler
        hard_candidates = [c for _, _, _, _, c in ranked[:HARD_NEG_CAND]]
        hard_set = set(hard_candidates)

        # easy pool: remaining exposed-not-clicked, then random refill
        easy_expo = sorted(
            [c for c in expo_unique if c not in hard_set],
            key=lambda x: expo_last_ts[x],
            reverse=True
        )

        easy_negs = easy_expo[:EASY_NEG_KEEP]
        easy_set = set(easy_negs)

        if len(easy_negs) < EASY_NEG_KEEP:
            refill_easy = self._random_refill(
                forbid,
                easy_set.union(hard_set),
                EASY_NEG_KEEP - len(easy_negs),
                generator
            )
            easy_negs.extend(refill_easy)

        if len(easy_negs) < EASY_NEG_KEEP:
            easy_negs += [0] * (EASY_NEG_KEEP - len(easy_negs))

        if len(hard_candidates) < HARD_NEG_CAND:
            hard_candidates += [0] * (HARD_NEG_CAND - len(hard_candidates))

        return (
            np.asarray(easy_negs[:EASY_NEG_KEEP], dtype=np.int64),
            np.asarray(hard_candidates[:HARD_NEG_CAND], dtype=np.int64)
        )

    def __getitem__(self, i):
        u_raw, click_pos = self.samples[i]
        d = self.user_event[u_raw]
        cp = d["click_pos"]

        idx = np.searchsorted(cp, click_pos)

        hist_click_pos = cp[:idx]
        if len(hist_click_pos) > self.max_hist:
            hist_click_pos = hist_click_pos[-self.max_hist:]

        uc = map_user_code(int(u_raw))
        target_pid = int(d["pid_code"][click_pos])

        # 8 fields: user + 6 demos + target_pid
        u_feats = np.concatenate(
            [
                np.array([uc], dtype=np.int64),
                d["demo"],
                np.array([target_pid], dtype=np.int64),
            ],
            axis=0
        )

        pos_raw = int(d["adg_raw"][click_pos])
        pos_adg = adg_id2idx.get(pos_raw, 0)

        if len(hist_click_pos) == 0:
            hist_adg = np.array([0], dtype=np.int64)
            hist_pid = np.array([0], dtype=np.int64)
            hist_gap = np.array([0], dtype=np.int64)
            hist_mon = np.array([0], dtype=np.int64)
        else:
            hist_adg_raw = d["adg_raw"][hist_click_pos].astype(np.int64)
            hist_adg = np.array([adg_id2idx.get(int(x), 0) for x in hist_adg_raw], dtype=np.int64)
            hist_pid = d["pid_code"][hist_click_pos].astype(np.int64)
            target_ts = int(d["ts"][click_pos])
            hist_ts = d["ts"][hist_click_pos].astype(np.int64)
            hist_gap = make_gap_bucket(target_ts, hist_ts).astype(np.int64)
            hist_mon = d["month"][hist_click_pos].astype(np.int64)

        easy_neg_adgs, hard_cand_adgs = self._sample_negatives(d, click_pos, pos_adg, i)

        pos_cate  = int(cate_by_adg[pos_adg]) if pos_adg else 0
        pos_brand = int(brand_by_adg[pos_adg]) if pos_adg else 0
        pos_price = float(price_by_adg[pos_adg]) if pos_adg else 0.0
        pos_item_cats = np.array([pos_adg, pos_cate, pos_brand], dtype=np.int64)

        hist_cate  = cate_by_adg[hist_adg]
        hist_brand = brand_by_adg[hist_adg]
        hist_price = price_by_adg[hist_adg].astype(np.float32)

        # 4 discrete history fields: adg, cate, brand, pid
        hist_item = np.stack([hist_adg, hist_cate, hist_brand, hist_pid], axis=1).astype(np.int64)
        hist_time = np.stack([hist_gap, hist_mon], axis=1).astype(np.int64)
        hist_cont = hist_price.reshape(-1, 1).astype(np.float32)

        lens = hist_item.shape[0]

        easy_neg_cate  = cate_by_adg[easy_neg_adgs]
        easy_neg_brand = brand_by_adg[easy_neg_adgs]
        easy_neg_price = price_by_adg[easy_neg_adgs].astype(np.float32)
        easy_neg_item_cats = np.stack(
            [easy_neg_adgs, easy_neg_cate, easy_neg_brand],
            axis=1
        ).astype(np.int64)

        hard_cand_cate  = cate_by_adg[hard_cand_adgs]
        hard_cand_brand = brand_by_adg[hard_cand_adgs]
        hard_cand_price = price_by_adg[hard_cand_adgs].astype(np.float32)
        hard_cand_item_cats = np.stack(
            [hard_cand_adgs, hard_cand_cate, hard_cand_brand],
            axis=1
        ).astype(np.int64)

        return (
            u_feats,
            hist_item,
            hist_time,
            hist_cont,
            pos_item_cats,
            np.float32(pos_price),
            easy_neg_item_cats,
            easy_neg_price,
            hard_cand_item_cats,
            hard_cand_price,
            lens
        )

def collate_fn(batch):
    (
        u_feats,
        hist_item,
        hist_time,
        hist_cont,
        pos_cats,
        pos_price,
        easy_neg_cats,
        easy_neg_price,
        hard_cand_cats,
        hard_cand_price,
        lens
    ) = zip(*batch)

    lens = torch.tensor(lens, dtype=torch.long)
    order = torch.argsort(lens, descending=True)
    lens = lens[order]

    u_feats = torch.tensor(np.stack(u_feats), dtype=torch.long)[order]
    pos_cats = torch.tensor(np.stack(pos_cats), dtype=torch.long)[order]
    pos_price = torch.tensor(np.array(pos_price, dtype=np.float32), dtype=torch.float32)[order]

    easy_neg_cats = torch.tensor(np.stack(easy_neg_cats), dtype=torch.long)[order]
    easy_neg_price = torch.tensor(np.stack(easy_neg_price), dtype=torch.float32)[order]

    hard_cand_cats = torch.tensor(np.stack(hard_cand_cats), dtype=torch.long)[order]
    hard_cand_price = torch.tensor(np.stack(hard_cand_price), dtype=torch.float32)[order]

    B = len(batch)
    Lmax = int(lens.max().item())

    hi_pad = torch.zeros((B, Lmax, 4), dtype=torch.long)
    ht_pad = torch.zeros((B, Lmax, 2), dtype=torch.long)
    hc_pad = torch.zeros((B, Lmax, 1), dtype=torch.float32)

    for bi, src_i in enumerate(order.tolist()):
        L = hist_item[src_i].shape[0]
        hi_pad[bi, :L] = torch.tensor(hist_item[src_i], dtype=torch.long)
        ht_pad[bi, :L] = torch.tensor(hist_time[src_i], dtype=torch.long)
        hc_pad[bi, :L] = torch.tensor(hist_cont[src_i], dtype=torch.float32)

    return (
        u_feats,
        hi_pad,
        ht_pad,
        hc_pad,
        pos_cats,
        pos_price,
        easy_neg_cats,
        easy_neg_price,
        hard_cand_cats,
        hard_cand_price,
        lens
    )

train_ds = ClickDatasetActive(
    user_event,
    "train",
    max_hist=MAX_HIST_CLICKS,
    deterministic=False,
    seed_base=1000
)
val_ds = ClickDatasetActive(
    user_event,
    "val",
    max_hist=MAX_HIST_CLICKS,
    deterministic=True,
    seed_base=2000
)
test_ds = ClickDatasetActive(
    user_event,
    "test",
    max_hist=MAX_HIST_CLICKS,
    deterministic=True,
    seed_base=3000
)

print("Samples active-only:", {
    "train": len(train_ds),
    "val": len(val_ds),
    "test": len(test_ds)
})

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    drop_last=True,
    collate_fn=collate_fn
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    collate_fn=collate_fn
)

test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    collate_fn=collate_fn
)

# -----------------------
# Model
# -----------------------
class ResidualMLP(nn.Module):
    def __init__(self, din, dh, dout, dropout=0.1):
        super().__init__()
        self.fc1 = nn.Linear(din, dh)
        self.fc2 = nn.Linear(dh, dout)
        self.ln  = nn.LayerNorm(dout)
        self.dp  = nn.Dropout(dropout)
        self.proj = nn.Identity() if din == dout else nn.Linear(din, dout)

    def forward(self, x):
        h = F.relu(self.fc1(x))
        h = self.dp(h)
        h = self.fc2(h)
        return self.ln(self.proj(x) + h)

class ItemTower(nn.Module):
    def __init__(self):
        super().__init__()

        self.adg_emb   = nn.Embedding(ADG_V,   D_ADG,   padding_idx=0)
        self.cate_emb  = nn.Embedding(CATE_V,  D_CATE,  padding_idx=0)
        self.brand_emb = nn.Embedding(BRAND_V, D_BRAND, padding_idx=0)

        in_dim = D_ADG + D_CATE + D_BRAND + 1
        self.m1 = ResidualMLP(in_dim, MLP_HIDDEN, MLP_HIDDEN, DROPOUT)
        self.m2 = ResidualMLP(MLP_HIDDEN, MLP_HIDDEN, ITEM_OUT_DIM, DROPOUT)

    def forward(self, item_cats_3, price_norm):
        adg, cate, brand = [item_cats_3[:, i] for i in range(3)]

        x = torch.cat([
            self.adg_emb(adg),
            self.cate_emb(cate),
            self.brand_emb(brand),
            price_norm.unsqueeze(-1),
        ], dim=-1)

        x = self.m1(x)
        x = self.m2(x)
        return F.normalize(x, dim=-1)

class SequenceBackbone(nn.Module):
    def __init__(self, mode: str, input_dim: int, hidden_dim: int, max_len: int):
        super().__init__()
        self.mode = mode
        self.max_len = max_len
        self.input_proj = nn.Linear(input_dim, hidden_dim)

        if mode == "gru":
            self.gru = nn.GRU(hidden_dim, hidden_dim, batch_first=True)

        elif mode == "transformer":
            self.pos_emb = nn.Embedding(max_len + 2, hidden_dim, padding_idx=0)
            enc_layer = nn.TransformerEncoderLayer(
                d_model=hidden_dim,
                nhead=TRANSFORMER_NHEAD,
                dim_feedforward=TRANSFORMER_FF_DIM,
                dropout=TRANSFORMER_DROPOUT,
                activation="gelu",
                batch_first=True,
                norm_first=True
            )
            self.transformer = nn.TransformerEncoder(enc_layer, num_layers=TRANSFORMER_LAYERS)
            self.out_ln = nn.LayerNorm(hidden_dim)

        elif mode == "gru_transformer":
            self.gru = nn.GRU(hidden_dim, hidden_dim, batch_first=True)
            self.pos_emb = nn.Embedding(max_len + 2, hidden_dim, padding_idx=0)
            enc_layer = nn.TransformerEncoderLayer(
                d_model=hidden_dim,
                nhead=TRANSFORMER_NHEAD,
                dim_feedforward=TRANSFORMER_FF_DIM,
                dropout=TRANSFORMER_DROPOUT,
                activation="gelu",
                batch_first=True,
                norm_first=True
            )
            self.transformer = nn.TransformerEncoder(enc_layer, num_layers=TRANSFORMER_LAYERS)
            self.out_ln = nn.LayerNorm(hidden_dim)
        else:
            raise ValueError(mode)

    def forward(self, seq_x, lens):
        B, L, _ = seq_x.shape
        zero_mask = lens <= 0
        safe_lens = lens.clamp(min=1)

        x = self.input_proj(seq_x)

        if self.mode == "gru":
            packed = pack_padded_sequence(x, safe_lens.cpu(), batch_first=True, enforce_sorted=False)
            _, hN = self.gru(packed)
            out = hN[-1]
            out[zero_mask] = 0.0
            return out

        pad_mask = torch.arange(L, device=seq_x.device).unsqueeze(0) >= safe_lens.unsqueeze(1)
        pos_ids = torch.arange(1, L + 1, device=seq_x.device).unsqueeze(0).expand(B, L)
        pos_ids = pos_ids.clamp(max=self.max_len + 1)

        if self.mode == "transformer":
            x = x + self.pos_emb(pos_ids)
            x = self.transformer(x, src_key_padding_mask=pad_mask)
            x = self.out_ln(x)
            last_idx = (safe_lens - 1).clamp(min=0)
            out = x[torch.arange(B, device=seq_x.device), last_idx]
            out[zero_mask] = 0.0
            return out

        packed = pack_padded_sequence(x, safe_lens.cpu(), batch_first=True, enforce_sorted=False)
        packed_out, _ = self.gru(packed)
        gru_out, _ = pad_packed_sequence(packed_out, batch_first=True, total_length=L)

        x = gru_out + self.pos_emb(pos_ids)
        x = self.transformer(x, src_key_padding_mask=pad_mask)
        x = self.out_ln(x)
        last_idx = (safe_lens - 1).clamp(min=0)
        out = x[torch.arange(B, device=seq_x.device), last_idx]
        out[zero_mask] = 0.0
        return out

class UserTower(nn.Module):
    def __init__(self, shared_item: ItemTower, history_encoder_mode: str = "gru"):
        super().__init__()

        self.user_emb = nn.Embedding(USER_V, D_USER, padding_idx=0)

        self.cms_seg_emb = nn.Embedding(CMS_SEG_V, D_CMS_SEG, padding_idx=0)
        self.cms_grp_emb = nn.Embedding(CMS_GRP_V, D_CMS_GRP, padding_idx=0)
        self.gender_emb  = nn.Embedding(GENDER_V,  D_GENDER,  padding_idx=0)
        self.age_emb     = nn.Embedding(AGE_V,     D_AGE,     padding_idx=0)
        self.shop_emb    = nn.Embedding(SHOP_V,    D_SHOP,    padding_idx=0)
        self.occ_emb     = nn.Embedding(OCC_V,     D_OCC,     padding_idx=0)

        self.pid_emb   = nn.Embedding(PID_V, D_PID, padding_idx=0)
        self.gap_emb   = nn.Embedding(TIME_GAP_V, D_GAP, padding_idx=0)
        self.month_emb = nn.Embedding(MONTH_V,    D_MONTH, padding_idx=0)

        self.adg_ev   = shared_item.adg_emb
        self.cate_ev  = shared_item.cate_emb
        self.brand_ev = shared_item.brand_emb

        seq_in_dim = D_ADG + D_CATE + D_BRAND + D_PID + D_GAP + D_MONTH + 1
        self.short_encoder = SequenceBackbone(history_encoder_mode, seq_in_dim, SEQ_HIDDEN, SHORT_HIST_CLICKS)
        self.long_encoder  = SequenceBackbone(history_encoder_mode, seq_in_dim, SEQ_HIDDEN, LONG_HIST_MAX)

        demo_dim = D_CMS_SEG + D_CMS_GRP + D_GENDER + D_AGE + D_SHOP + D_OCC
        ctx_dim = D_USER + demo_dim + D_PID

        self.fuse_gate = nn.Linear(SEQ_HIDDEN * 2 + ctx_dim, SEQ_HIDDEN)

        head_in = SEQ_HIDDEN + ctx_dim
        self.m1 = ResidualMLP(head_in, MLP_HIDDEN, MLP_HIDDEN, DROPOUT)
        self.m2 = ResidualMLP(MLP_HIDDEN, MLP_HIDDEN, ITEM_OUT_DIM, DROPOUT)

    def _split_short_long(self, ev, lens):
        B, L, D = ev.shape

        short_ev = ev.new_zeros((B, SHORT_HIST_CLICKS, D))
        long_ev  = ev.new_zeros((B, LONG_HIST_MAX, D))

        short_lens = torch.clamp(lens, max=SHORT_HIST_CLICKS)
        long_lens  = torch.clamp(lens - SHORT_HIST_CLICKS, min=0, max=LONG_HIST_MAX)

        for i in range(B):
            li = int(lens[i].item())
            s_len = int(short_lens[i].item())
            l_len = int(long_lens[i].item())

            if s_len > 0:
                short_ev[i, :s_len] = ev[i, li - s_len:li]

            if l_len > 0:
                long_end = max(li - SHORT_HIST_CLICKS, 0)
                long_start = max(long_end - l_len, 0)
                long_ev[i, :l_len] = ev[i, long_start:long_end]

        return short_ev, short_lens, long_ev, long_lens

    def forward(self, u_feats, hi_pad, ht_pad, hc_pad, lens):
        # 8 fields now
        u_code, cms_seg, cms_grp, gender, age, shop, occ, target_pid = [u_feats[:, i] for i in range(8)]

        e_u = self.user_emb(u_code)
        e_target_pid = self.pid_emb(target_pid)

        e_demo = torch.cat([
            self.cms_seg_emb(cms_seg),
            self.cms_grp_emb(cms_grp),
            self.gender_emb(gender),
            self.age_emb(age),
            self.shop_emb(shop),
            self.occ_emb(occ),
        ], dim=-1)

        hadg   = hi_pad[:, :, 0]
        hcate  = hi_pad[:, :, 1]
        hbrand = hi_pad[:, :, 2]
        hpid   = hi_pad[:, :, 3]

        hgap, hmonth = [ht_pad[:, :, i] for i in range(2)]
        hprice = hc_pad[:, :, 0]

        ev = torch.cat([
            self.adg_ev(hadg),
            self.cate_ev(hcate),
            self.brand_ev(hbrand),
            self.pid_emb(hpid),
            self.gap_emb(hgap),
            self.month_emb(hmonth),
            hprice.unsqueeze(-1),
        ], dim=-1)

        short_ev, short_lens, long_ev, long_lens = self._split_short_long(ev, lens)
        short_h = self.short_encoder(short_ev, short_lens)
        long_h  = self.long_encoder(long_ev, long_lens)

        ctx = torch.cat([e_u, e_demo, e_target_pid], dim=-1)

        gate_in = torch.cat([short_h, long_h, ctx], dim=-1)
        g = torch.sigmoid(self.fuse_gate(gate_in))
        h_seq = g * short_h + (1.0 - g) * long_h

        x = torch.cat([h_seq, ctx], dim=-1)
        x = self.m1(x)
        x = self.m2(x)
        return F.normalize(x, dim=-1)

class TwoTowerSeq(nn.Module):
    def __init__(self, history_encoder_mode: str = "gru"):
        super().__init__()
        self.item = ItemTower()
        self.user = UserTower(self.item, history_encoder_mode=history_encoder_mode)

    def user_vec(self, u_feats, hi_pad, ht_pad, hc_pad, lens):
        return self.user(u_feats, hi_pad, ht_pad, hc_pad, lens)

    def item_vec(self, item_cats_3, price_norm):
        return self.item(item_cats_3, price_norm)

model = TwoTowerSeq(history_encoder_mode=HISTORY_ENCODER).to(DEVICE)
print("History encoder mode:", HISTORY_ENCODER)

# -----------------------
# AdamW param groups
# -----------------------
def build_adamw_param_groups(model, weight_decay):
    decay_params = []
    no_decay_params = []

    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue

        lname = name.lower()
        if p.ndim == 1 or lname.endswith("bias") or "norm" in lname or "emb" in lname:
            no_decay_params.append(p)
        else:
            decay_params.append(p)

    return [
        {"params": decay_params, "weight_decay": weight_decay},
        {"params": no_decay_params, "weight_decay": 0.0},
    ]

opt = torch.optim.AdamW(
    build_adamw_param_groups(model, WEIGHT_DECAY),
    lr=LR
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    opt,
    mode="max",
    factor=LR_SCHED_FACTOR,
    patience=LR_SCHED_PATIENCE,
    min_lr=MIN_LR
)

# -----------------------
# Temperature schedule
# -----------------------
def tau_schedule(epoch, step, steps_per_epoch):
    if TAU_WARMUP_EPOCHS > 0 and epoch < TAU_WARMUP_EPOCHS:
        t = (epoch * steps_per_epoch + step) / max(TAU_WARMUP_EPOCHS * steps_per_epoch, 1)
        return TAU_START + t * (TAU_END - TAU_START)
    return TAU_END

# -----------------------
# Loss helpers
# -----------------------
def infonce_inbatch_masked(u_vec, p_vec, pos_adg, tau):
    logits = (u_vec @ p_vec.t()) / tau
    labels = torch.arange(u_vec.size(0), device=u_vec.device)

    same_item = pos_adg.unsqueeze(0).eq(pos_adg.unsqueeze(1))
    eye = torch.eye(pos_adg.size(0), device=pos_adg.device, dtype=torch.bool)
    dup_neg_mask = same_item & (~eye)

    logits = logits.masked_fill(dup_neg_mask, -1e9)
    return F.cross_entropy(logits, labels)

def select_ohem_hard_negatives(u_vec, hard_cand_vec, hard_valid_mask, keep_k):
    B, C, D = hard_cand_vec.shape

    if keep_k <= 0:
        return hard_cand_vec[:, :0], hard_valid_mask[:, :0]

    scores = (u_vec.unsqueeze(1) * hard_cand_vec).sum(dim=-1)
    scores = scores.masked_fill(~hard_valid_mask, -1e9)

    top_idx = torch.topk(
        scores,
        k=min(keep_k, C),
        dim=1,
        largest=True,
        sorted=False
    ).indices

    gather_idx = top_idx.unsqueeze(-1).expand(-1, -1, D)
    selected_vec = torch.gather(hard_cand_vec, 1, gather_idx)
    selected_valid = torch.gather(hard_valid_mask, 1, top_idx)

    return selected_vec, selected_valid

def infonce_sampled(u_vec, p_vec, n_vecK, neg_valid_mask, tau):
    pos_score = (u_vec * p_vec).sum(dim=-1, keepdim=True)
    neg_score = (u_vec.unsqueeze(1) * n_vecK).sum(dim=-1)
    neg_score = neg_score.masked_fill(~neg_valid_mask, -1e9)

    logits = torch.cat([pos_score, neg_score], dim=1) / tau
    labels = torch.zeros(u_vec.size(0), device=u_vec.device, dtype=torch.long)

    return F.cross_entropy(logits, labels)

def build_sampled_negatives_for_loss(
    model,
    u_vec,
    easy_neg_cats,
    easy_neg_price,
    hard_cand_cats,
    hard_cand_price
):
    B = u_vec.size(0)
    D = ITEM_OUT_DIM

    # easy negatives
    Be, Ke, _ = easy_neg_cats.shape
    assert Be == B
    easy_valid = (easy_neg_cats[:, :, 0] != 0)

    easy_vec = model.item_vec(
        easy_neg_cats.reshape(B * Ke, 3),
        easy_neg_price.reshape(B * Ke)
    ).view(B, Ke, D)

    # hard candidates from exposed-not-clicked only
    Bh, Kh, _ = hard_cand_cats.shape
    assert Bh == B
    hard_valid = (hard_cand_cats[:, :, 0] != 0)

    hard_cand_vec = model.item_vec(
        hard_cand_cats.reshape(B * Kh, 3),
        hard_cand_price.reshape(B * Kh)
    ).view(B, Kh, D)

    selected_hard_vec, selected_hard_valid = select_ohem_hard_negatives(
        u_vec,
        hard_cand_vec,
        hard_valid,
        keep_k=HARD_NEG_KEEP
    )

    neg_vec = torch.cat([easy_vec, selected_hard_vec], dim=1)
    neg_valid = torch.cat([easy_valid, selected_hard_valid], dim=1)

    return neg_vec, neg_valid

def total_infonce_loss(
    model,
    u_vec,
    p_vec,
    pos_adg,
    easy_neg_cats,
    easy_neg_price,
    hard_cand_cats,
    hard_cand_price,
    tau
):
    lossA = infonce_inbatch_masked(u_vec, p_vec, pos_adg, tau=tau)

    neg_vec, neg_valid = build_sampled_negatives_for_loss(
        model,
        u_vec,
        easy_neg_cats,
        easy_neg_price,
        hard_cand_cats,
        hard_cand_price
    )

    lossB = infonce_sampled(
        u_vec,
        p_vec,
        neg_vec,
        neg_valid,
        tau=tau
    )

    loss = LOSS_A_MULT * lossA + LOSS_B_MULT * lossB
    return loss, lossA, lossB

@torch.no_grad()
def val_loss_fixed_tau(model, loader, tau=TAU_VAL):
    model.eval()

    tot, n = 0.0, 0

    for batch in loader:
        (
            u_feats, hi_pad, ht_pad, hc_pad, pos_cats, pos_price,
            easy_neg_cats, easy_neg_price, hard_cand_cats, hard_cand_price, lens
        ) = batch

        u_feats = u_feats.to(DEVICE)
        hi_pad = hi_pad.to(DEVICE)
        ht_pad = ht_pad.to(DEVICE)
        hc_pad = hc_pad.to(DEVICE)

        pos_cats = pos_cats.to(DEVICE)
        pos_price = pos_price.to(DEVICE)

        easy_neg_cats = easy_neg_cats.to(DEVICE)
        easy_neg_price = easy_neg_price.to(DEVICE)

        hard_cand_cats = hard_cand_cats.to(DEVICE)
        hard_cand_price = hard_cand_price.to(DEVICE)

        lens = lens.to(DEVICE)

        u = model.user_vec(u_feats, hi_pad, ht_pad, hc_pad, lens)
        p = model.item_vec(pos_cats, pos_price)
        pos_adg = pos_cats[:, 0]

        loss, lossA, lossB = total_infonce_loss(
            model,
            u,
            p,
            pos_adg,
            easy_neg_cats,
            easy_neg_price,
            hard_cand_cats,
            hard_cand_price,
            tau=tau
        )

        B = u.size(0)
        tot += float(loss.item()) * B
        n += B

    return tot / max(n, 1)

# -----------------------
# Active-pool recall@K
# -----------------------
@torch.no_grad()
def build_item_matrix_active(model, chunk=65536):
    model.eval()

    codes = torch.from_numpy(ACTIVE_ADG_CODES).to(device=DEVICE, dtype=torch.long)
    N = int(codes.numel())

    cate = torch.from_numpy(cate_by_adg[ACTIVE_ADG_CODES]).to(device=DEVICE, dtype=torch.long)
    brand = torch.from_numpy(brand_by_adg[ACTIVE_ADG_CODES]).to(device=DEVICE, dtype=torch.long)
    price = torch.from_numpy(price_by_adg[ACTIVE_ADG_CODES]).to(device=DEVICE, dtype=torch.float32)

    mat = torch.empty((N, ITEM_OUT_DIM), device=DEVICE, dtype=torch.float32)

    for s in range(0, N, chunk):
        e = min(N, s + chunk)
        cats = torch.stack([codes[s:e], cate[s:e], brand[s:e]], dim=1)
        mat[s:e] = model.item_vec(cats, price[s:e])

    return codes, mat

@torch.no_grad()
def recall_at_k_active(
    model,
    loader,
    active_codes,
    active_mat,
    Ks=(1, 5, 10, 20, 50, 100, 200, 500, 1000)
):
    model.eval()

    Ks = sorted(Ks)
    maxK = min(Ks[-1], active_mat.size(0))
    Ks = [k for k in Ks if k <= maxK]

    hits = {k: 0 for k in Ks}
    total = 0

    code2row = -np.ones((ADG_V,), dtype=np.int64)
    code2row[active_codes.detach().cpu().numpy()] = np.arange(active_codes.numel(), dtype=np.int64)

    active_T = active_mat.transpose(0, 1).contiguous()

    for batch in loader:
        (
            u_feats, hi_pad, ht_pad, hc_pad, pos_cats, pos_price,
            easy_neg_cats, easy_neg_price, hard_cand_cats, hard_cand_price, lens
        ) = batch

        u_feats = u_feats.to(DEVICE)
        hi_pad = hi_pad.to(DEVICE)
        ht_pad = ht_pad.to(DEVICE)
        hc_pad = hc_pad.to(DEVICE)
        lens = lens.to(DEVICE)

        uvec = model.user_vec(u_feats, hi_pad, ht_pad, hc_pad, lens)

        pos_adg = pos_cats[:, 0].numpy().astype(np.int64)
        true_row = code2row[pos_adg]
        true_row = torch.from_numpy(true_row).to(device=DEVICE, dtype=torch.long)

        scores = uvec @ active_T

        top_idx = torch.topk(
            scores,
            k=maxK,
            dim=1,
            largest=True,
            sorted=False
        ).indices

        hit_mat = top_idx == true_row.unsqueeze(1)

        for k in Ks:
            hits[k] += int(hit_mat[:, :k].any(dim=1).sum().item())

        total += pos_adg.shape[0]

    return {f"recall@{k}": hits[k] / max(total, 1) for k in Ks}

# -----------------------
# Train
# -----------------------
steps_per_epoch = max(len(train_loader), 1)

best_metric = -float("inf")
best_epoch = -1
best_state = copy.deepcopy(model.state_dict())
epochs_no_improve = 0

for epoch in range(EPOCHS):
    model.train()

    running, seen = 0.0, 0

    for step, batch in enumerate(train_loader, start=0):
        (
            u_feats, hi_pad, ht_pad, hc_pad, pos_cats, pos_price,
            easy_neg_cats, easy_neg_price, hard_cand_cats, hard_cand_price, lens
        ) = batch

        u_feats = u_feats.to(DEVICE)
        hi_pad = hi_pad.to(DEVICE)
        ht_pad = ht_pad.to(DEVICE)
        hc_pad = hc_pad.to(DEVICE)

        pos_cats = pos_cats.to(DEVICE)
        pos_price = pos_price.to(DEVICE)

        easy_neg_cats = easy_neg_cats.to(DEVICE)
        easy_neg_price = easy_neg_price.to(DEVICE)

        hard_cand_cats = hard_cand_cats.to(DEVICE)
        hard_cand_price = hard_cand_price.to(DEVICE)

        lens = lens.to(DEVICE)

        tau = tau_schedule(epoch, step, steps_per_epoch)

        opt.zero_grad(set_to_none=True)

        u = model.user_vec(u_feats, hi_pad, ht_pad, hc_pad, lens)
        p = model.item_vec(pos_cats, pos_price)
        pos_adg = pos_cats[:, 0]

        loss, lossA, lossB = total_infonce_loss(
            model,
            u,
            p,
            pos_adg,
            easy_neg_cats,
            easy_neg_price,
            hard_cand_cats,
            hard_cand_price,
            tau=tau
        )

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        opt.step()

        B = u.size(0)
        running += float(loss.item()) * B
        seen += B

        if (step + 1) % 50 == 0:
            current_lr = opt.param_groups[0]["lr"]
            print(
                f"Epoch {epoch} step {step + 1}: "
                f"tau={tau:.4f} lr={current_lr:.2e} "
                f"loss={running / seen:.4f} "
                f"(masked_inbatch={lossA.item():.4f}, "
                f"sampled_OHEM_soft={lossB.item():.4f})"
            )

    train_loss = running / max(seen, 1)
    vloss = val_loss_fixed_tau(model, val_loader, tau=TAU_VAL)

    active_codes, active_mat = build_item_matrix_active(model)

    vrec = recall_at_k_active(
        model,
        val_loader,
        active_codes,
        active_mat,
        Ks=(1, 5, 10, 20, 50, 100, 200, 500, 1000)
    )

    monitor_value = vrec.get(MONITOR_METRIC, float("-inf"))
    scheduler.step(monitor_value)
    current_lr = opt.param_groups[0]["lr"]

    improved = monitor_value > best_metric + 1e-6
    if improved:
        best_metric = monitor_value
        best_epoch = epoch
        best_state = copy.deepcopy(model.state_dict())
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1

    print(
        f"\nEpoch {epoch} done: "
        f"train_loss={train_loss:.4f} "
        f"val_loss={vloss:.4f} "
        f"monitor {MONITOR_METRIC}={monitor_value:.4f} "
        f"best_epoch={best_epoch} best_{MONITOR_METRIC}={best_metric:.4f} "
        f"lr={current_lr:.2e} "
        f"r@5={vrec.get('recall@5', float('nan')):.4f} "
        f"r@10={vrec.get('recall@10', float('nan')):.4f} "
        f"r@20={vrec.get('recall@20', float('nan')):.4f} "
        f"r@50={vrec.get('recall@50', float('nan')):.4f} "
        f"r@100={vrec.get('recall@100', float('nan')):.4f} "
        f"r@200={vrec.get('recall@200', float('nan')):.4f} "
        f"r@500={vrec.get('recall@500', float('nan')):.4f} "
        f"r@1000={vrec.get('recall@1000', float('nan')):.4f}\n"
    )

    if epochs_no_improve >= EARLY_STOP_PATIENCE:
        print(
            f"Early stopping triggered at epoch {epoch}. "
            f"Best epoch={best_epoch}, best {MONITOR_METRIC}={best_metric:.4f}"
        )
        break

# -----------------------
# Restore best checkpoint
# -----------------------
model.load_state_dict(best_state)
print(f"Restored best checkpoint from epoch {best_epoch} with {MONITOR_METRIC}={best_metric:.4f}")

# -----------------------
# Final test
# -----------------------
active_codes, active_mat = build_item_matrix_active(model)

trec = recall_at_k_active(
    model,
    test_loader,
    active_codes,
    active_mat,
    Ks=(1, 5, 10, 20, 50, 100, 200, 500, 1000)
)

tloss = val_loss_fixed_tau(
    model,
    test_loader,
    tau=TAU_VAL
)

print("TEST ACTIVE POOL Recall:", trec)
print("TEST loss const tau:", tloss)

DEVICE: cuda
/content/taobao_ad/ 100%[===================>]  29.84M  84.6MB/s    in 0.4s    


/tmp/ipykernel_3260/2559374509.py:63: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(os.path.dirname(path))


/content/taobao_ad/ 100%[===================>]   1.01G   147MB/s    in 7.2s    
/content/taobao_ad/ 100%[===================>]  22.95M  76.0MB/s    in 0.3s    
Files ready
{'TOPK_CAP_USER': 400000, 'USER_HASH_BUCKETS': 200000, 'K_NEG': 255, 'EASY_NEG_KEEP': 229, 'HARD_NEG_KEEP': 26, 'HARD_NEG_CAND': 104, 'LOSS_A_MULT': 0.5, 'LOSS_B_MULT': 3.0, 'BATCH_SIZE': 768, 'MONITOR_METRIC': 'recall@50'}
raw rows: 26557961
USER_TOPK chosen=400000 covers click volume=0.9329 target=0.95
ADG_TOPK  chosen=171828 covers click volume=0.9500 target=0.95
Vocabs: {'USER_V': 600001, 'ADG_V': 171829, 'CATE_V': 6770, 'BRAND_V': 2001, 'PID_V': 3}
Embedding dims: {'user': 48, 'adg': 64, 'cate': 16, 'brand': 16, 'pid': 8, 'cms_seg': 8, 'cms_grp': 8, 'gender': 8, 'age': 8, 'shop': 8, 'occ': 8, 'gap': 8, 'month': 8}


building user_event: 100%|██████████| 1141729/1141729 [00:51<00:00, 22272.66it/s]


Users kept: 149936
Kept users covered explicitly by top user vocab: 149936/149936 = 1.0000
[POOL] min_recent= 50 pool=  3385 val_cov=0.319 (47849/149936) test_cov=0.317 (47597/149936)
[POOL] min_recent= 30 pool=  6828 val_cov=0.414 (62120/149936) test_cov=0.412 (61709/149936)
[POOL] min_recent= 18 pool= 13297 val_cov=0.521 (78099/149936) test_cov=0.518 (77623/149936)
[POOL] min_recent= 10 pool= 26494 val_cov=0.643 (96429/149936) test_cov=0.640 (95921/149936)
[POOL] min_recent=  6 pool= 46075 val_cov=0.746 (111873/149936) test_cov=0.741 (111133/149936)
[POOL] min_recent=  3 pool= 91000 val_cov=0.865 (129733/149936) test_cov=0.862 (129247/149936)
[POOL] min_recent=  1 pool=171828 val_cov=0.951 (142596/149936) test_cov=0.952 (142690/149936)
Final active pool: size=171828, min_recent=1, val_cov=0.951, test_cov=0.952, end_ts=1494691186, cut_ts=1493481586
Popularity truncation cap=127.865
Samples active-only: {'train': 588294, 'val': 142596, 'test': 142690}


/tmp/ipykernel_3260/2559374509.py:1044: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(enc_layer, num_layers=TRANSFORMER_LAYERS)


History encoder mode: gru_transformer
Epoch 0 step 50: tau=0.1983 lr=2.00e-04 loss=19.8734 (masked_inbatch=6.6359, sampled_OHEM_soft=5.4767)
Epoch 0 step 100: tau=0.1966 lr=2.00e-04 loss=19.7326 (masked_inbatch=6.5588, sampled_OHEM_soft=5.3876)
Epoch 0 step 150: tau=0.1949 lr=2.00e-04 loss=19.6205 (masked_inbatch=6.4572, sampled_OHEM_soft=5.3073)
Epoch 0 step 200: tau=0.1932 lr=2.00e-04 loss=19.5138 (masked_inbatch=6.3943, sampled_OHEM_soft=5.2889)
Epoch 0 step 250: tau=0.1915 lr=2.00e-04 loss=19.4199 (masked_inbatch=6.3306, sampled_OHEM_soft=5.2404)
Epoch 0 step 300: tau=0.1899 lr=2.00e-04 loss=19.3314 (masked_inbatch=6.4250, sampled_OHEM_soft=5.3073)
Epoch 0 step 350: tau=0.1882 lr=2.00e-04 loss=19.2572 (masked_inbatch=6.3222, sampled_OHEM_soft=5.2409)
Epoch 0 step 400: tau=0.1865 lr=2.00e-04 loss=19.1889 (masked_inbatch=6.2215, sampled_OHEM_soft=5.1825)
Epoch 0 step 450: tau=0.1848 lr=2.00e-04 loss=19.1312 (masked_inbatch=6.2734, sampled_OHEM_soft=5.2173)
Epoch 0 step 500: tau=0.183

In [ ]:

# ============================================================
# MULTI-ROUTE RETRIEVAL + XGBOOST RERANK
#
# Assumes the previous dual-tower training cell has already run
# and the following objects already exist in memory:
#   model, user_event, adg_id2idx, map_user_code,
#   cate_by_adg, brand_by_adg, price_by_adg,
#   total_clicks, recent_clicks, ACTIVE_ADG_CODES, active_mask,
#   MAX_HIST_CLICKS
#
# This cell implements:
#   1) DSSM / dual-tower retrieval
#   2) ItemCF retrieval
#   3) Hot retrieval
#   4) candidate union
#   5) XGBRanker reranking
#   6) val/test evaluation
# ============================================================

!pip -q install -U "xgboost>=2.0"

import math
import random
from collections import Counter, defaultdict

import numpy as np
import torch
import xgboost as xgb
from tqdm import tqdm

# -----------------------
# Config
# -----------------------
SEED = 42
rng = random.Random(SEED)
np.random.seed(SEED)

# query subsampling for practicality
MAX_RERANK_TRAIN_QUERIES = 3000
MAX_RERANK_VAL_QUERIES   = 1000
MAX_RERANK_TEST_QUERIES  = 3000

# route sizes
DT_ROUTE_TOPK  = 300
CF_ROUTE_TOPK  = 300
HOT_ROUTE_TOPK = 100

# ItemCF configs
CF_RECENT_CLICKS = 10
CF_WINDOW = 3
CF_TOPN_PER_ITEM = 80
CF_QUERY_RECENT = 5

# XGBoost configs
XGB_NUM_BOOST_ROUND = 200
XGB_LR = 0.1
XGB_MAX_DEPTH = 6
XGB_SUBSAMPLE = 0.8
XGB_COLSAMPLE = 0.8

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)

# -----------------------
# Sanity checks
# -----------------------
required_names = [
    "model", "user_event", "adg_id2idx", "map_user_code",
    "cate_by_adg", "brand_by_adg", "price_by_adg",
    "total_clicks", "recent_clicks", "ACTIVE_ADG_CODES", "active_mask",
    "MAX_HIST_CLICKS", "TIME_GAP_BUCKETS"
]
missing = [name for name in required_names if name not in globals()]
if missing:
    raise RuntimeError(f"Missing required objects from previous cell: {missing}")

model.eval()

# -----------------------
# Helper: time gap bucket
# -----------------------
def make_gap_bucket(target_ts: int, hist_ts: np.ndarray):
    delta_sec = np.maximum(target_ts - hist_ts, 0)
    delta_day = delta_sec.astype(np.float64) / 86400.0
    bucket = np.floor(np.log2(delta_day + 1.0)).astype(np.int64)
    bucket = np.clip(bucket, 0, TIME_GAP_BUCKETS - 1)
    return bucket

# -----------------------
# Build query lists
# -----------------------
def build_query_tuples(user_event_dict, split: str):
    queries = []
    for u, d in user_event_dict.items():
        cp = d["click_pos"]

        if split == "train":
            use = cp[:-2]
        elif split == "val":
            use = cp[-2:-1]
        elif split == "test":
            use = cp[-1:]
        else:
            raise ValueError(split)

        for p in use:
            pos_raw = int(d["adg_raw"][int(p)])
            pos_code = adg_id2idx.get(pos_raw, 0)
            if pos_code == 0 or not active_mask[pos_code]:
                continue
            queries.append((int(u), int(p)))
    return queries

train_queries_all = build_query_tuples(user_event, "train")
val_queries_all   = build_query_tuples(user_event, "val")
test_queries_all  = build_query_tuples(user_event, "test")

print({
    "train_queries_all": len(train_queries_all),
    "val_queries_all": len(val_queries_all),
    "test_queries_all": len(test_queries_all),
})

def sample_queries(query_list, limit, seed=SEED):
    if limit is None or len(query_list) <= limit:
        return list(query_list)
    rr = random.Random(seed)
    return rr.sample(query_list, limit)

train_queries = sample_queries(train_queries_all, MAX_RERANK_TRAIN_QUERIES, seed=SEED + 1)
val_queries   = sample_queries(val_queries_all,   MAX_RERANK_VAL_QUERIES,   seed=SEED + 2)
test_queries  = sample_queries(test_queries_all,  MAX_RERANK_TEST_QUERIES,  seed=SEED + 3)

print({
    "train_queries_used": len(train_queries),
    "val_queries_used": len(val_queries),
    "test_queries_used": len(test_queries),
})

# -----------------------
# Build query examples
# -----------------------
def make_query_example(u_raw, click_pos):
    d = user_event[u_raw]
    cp = d["click_pos"]

    idx = int(np.searchsorted(cp, click_pos))

    # clicked before target (all previous clicks)
    clicked_before_pos = cp[:idx]
    clicked_before_codes = np.array(
        [adg_id2idx.get(int(x), 0) for x in d["adg_raw"][clicked_before_pos].tolist()],
        dtype=np.int64
    )
    clicked_before_codes = clicked_before_codes[clicked_before_codes != 0]

    # last MAX_HIST_CLICKS for sequence encoder
    hist_click_pos = clicked_before_pos
    if len(hist_click_pos) > MAX_HIST_CLICKS:
        hist_click_pos = hist_click_pos[-MAX_HIST_CLICKS:]

    uc = map_user_code(int(u_raw))
    target_pid = int(d["pid_code"][click_pos])

    u_feats = np.concatenate(
        [
            np.array([uc], dtype=np.int64),
            d["demo"],
            np.array([target_pid], dtype=np.int64),
        ],
        axis=0
    )

    pos_raw = int(d["adg_raw"][click_pos])
    pos_adg = adg_id2idx.get(pos_raw, 0)

    if len(hist_click_pos) == 0:
        hist_adg = np.array([0], dtype=np.int64)
        hist_pid = np.array([0], dtype=np.int64)
        hist_gap = np.array([0], dtype=np.int64)
        hist_mon = np.array([0], dtype=np.int64)
    else:
        hist_adg_raw = d["adg_raw"][hist_click_pos].astype(np.int64)
        hist_adg = np.array([adg_id2idx.get(int(x), 0) for x in hist_adg_raw], dtype=np.int64)
        hist_pid = d["pid_code"][hist_click_pos].astype(np.int64)

        target_ts = int(d["ts"][click_pos])
        hist_ts = d["ts"][hist_click_pos].astype(np.int64)
        hist_gap = make_gap_bucket(target_ts, hist_ts).astype(np.int64)
        hist_mon = d["month"][hist_click_pos].astype(np.int64)

    hist_cate  = cate_by_adg[hist_adg]
    hist_brand = brand_by_adg[hist_adg]
    hist_price = price_by_adg[hist_adg].astype(np.float32)

    hist_item = np.stack([hist_adg, hist_cate, hist_brand, hist_pid], axis=1).astype(np.int64)
    hist_time = np.stack([hist_gap, hist_mon], axis=1).astype(np.int64)
    hist_cont = hist_price.reshape(-1, 1).astype(np.float32)

    last_adg = int(hist_adg[-1]) if len(hist_adg) > 0 else 0
    last_cate = int(hist_cate[-1]) if len(hist_cate) > 0 else 0
    last_brand = int(hist_brand[-1]) if len(hist_brand) > 0 else 0
    last_price = float(hist_price[-1]) if len(hist_price) > 0 else 0.0

    return {
        "u_raw": int(u_raw),
        "click_pos": int(click_pos),
        "u_feats": u_feats,
        "hist_item": hist_item,
        "hist_time": hist_time,
        "hist_cont": hist_cont,
        "lens": int(hist_item.shape[0]),
        "pos_adg": int(pos_adg),
        "target_pid": int(target_pid),

        "clicked_before_codes": clicked_before_codes,
        "hist_adg": hist_adg,
        "hist_cate": hist_cate,
        "hist_brand": hist_brand,
        "hist_price": hist_price,

        "last_adg": last_adg,
        "last_cate": last_cate,
        "last_brand": last_brand,
        "last_price": last_price,
    }

def build_examples(query_tuples, desc):
    examples = []
    for u_raw, click_pos in tqdm(query_tuples, desc=desc):
        ex = make_query_example(u_raw, click_pos)
        examples.append(ex)
    return examples

train_examples = build_examples(train_queries, "building train query examples")
val_examples   = build_examples(val_queries,   "building val query examples")
test_examples  = build_examples(test_queries,  "building test query examples")

# -----------------------
# Build active item matrix for dual-tower route
# -----------------------
@torch.no_grad()
def build_item_matrix_active(model, chunk=65536):
    model.eval()

    codes = torch.from_numpy(ACTIVE_ADG_CODES).to(device=DEVICE, dtype=torch.long)
    N = int(codes.numel())

    cate = torch.from_numpy(cate_by_adg[ACTIVE_ADG_CODES]).to(device=DEVICE, dtype=torch.long)
    brand = torch.from_numpy(brand_by_adg[ACTIVE_ADG_CODES]).to(device=DEVICE, dtype=torch.long)
    price = torch.from_numpy(price_by_adg[ACTIVE_ADG_CODES]).to(device=DEVICE, dtype=torch.float32)

    mat = torch.empty((N, model.item.m2.ln.normalized_shape[0]), device=DEVICE, dtype=torch.float32)

    for s in range(0, N, chunk):
        e = min(N, s + chunk)
        cats = torch.stack([codes[s:e], cate[s:e], brand[s:e]], dim=1)
        mat[s:e] = model.item_vec(cats, price[s:e])

    return codes, mat

active_codes_t, active_mat_t = build_item_matrix_active(model)
active_T_t = active_mat_t.transpose(0, 1).contiguous()

print("active item matrix:", tuple(active_mat_t.shape))

# -----------------------
# Query collate for DSSM route
# -----------------------
def collate_query_examples(examples):
    B = len(examples)
    Lmax = max(ex["lens"] for ex in examples)

    u_feats = torch.tensor(np.stack([ex["u_feats"] for ex in examples]), dtype=torch.long)
    hi_pad = torch.zeros((B, Lmax, 4), dtype=torch.long)
    ht_pad = torch.zeros((B, Lmax, 2), dtype=torch.long)
    hc_pad = torch.zeros((B, Lmax, 1), dtype=torch.float32)
    lens = torch.tensor([ex["lens"] for ex in examples], dtype=torch.long)

    for i, ex in enumerate(examples):
        L = ex["lens"]
        hi_pad[i, :L] = torch.tensor(ex["hist_item"], dtype=torch.long)
        ht_pad[i, :L] = torch.tensor(ex["hist_time"], dtype=torch.long)
        hc_pad[i, :L] = torch.tensor(ex["hist_cont"], dtype=torch.float32)

    return u_feats, hi_pad, ht_pad, hc_pad, lens

@torch.no_grad()
def dual_tower_route(examples, topk=DT_ROUTE_TOPK, batch_size=256):
    model.eval()
    results = []

    for s in tqdm(range(0, len(examples), batch_size), desc=f"DSSM route top{topk}"):
        batch_ex = examples[s:s+batch_size]
        u_feats, hi_pad, ht_pad, hc_pad, lens = collate_query_examples(batch_ex)

        u_feats = u_feats.to(DEVICE)
        hi_pad = hi_pad.to(DEVICE)
        ht_pad = ht_pad.to(DEVICE)
        hc_pad = hc_pad.to(DEVICE)
        lens = lens.to(DEVICE)

        uvec = model.user_vec(u_feats, hi_pad, ht_pad, hc_pad, lens)
        scores = uvec @ active_T_t

        top_scores, top_idx = torch.topk(
            scores,
            k=min(topk, scores.size(1)),
            dim=1,
            largest=True,
            sorted=True
        )

        top_codes = active_codes_t[top_idx]

        top_scores = top_scores.detach().cpu().numpy()
        top_codes = top_codes.detach().cpu().numpy()

        for i in range(top_codes.shape[0]):
            route = [(int(c), float(sc)) for c, sc in zip(top_codes[i], top_scores[i])]
            results.append(route)

    return results

# -----------------------
# Build ItemCF + hot route statistics from TRAIN split only
# -----------------------
def build_itemcf_and_hot(user_event_dict):
    item_cnt = Counter()
    co_counts = defaultdict(lambda: defaultdict(float))
    global_hot = Counter()
    pid_hot = defaultdict(Counter)

    for u, d in tqdm(user_event_dict.items(), desc="building ItemCF + hot stats"):
        cp_train = d["click_pos"][:-2]
        if len(cp_train) == 0:
            continue

        seq_codes = []
        seq_pids = []

        for p in cp_train:
            code = adg_id2idx.get(int(d["adg_raw"][p]), 0)
            if code == 0 or not active_mask[code]:
                continue
            seq_codes.append(int(code))
            seq_pids.append(int(d["pid_code"][p]))

        if len(seq_codes) == 0:
            continue

        seq_codes = seq_codes[-CF_RECENT_CLICKS:]
        seq_pids = seq_pids[-CF_RECENT_CLICKS:]

        for item, pid in zip(seq_codes, seq_pids):
            item_cnt[item] += 1
            global_hot[item] += 1
            pid_hot[pid][item] += 1

        n = len(seq_codes)
        for i in range(n):
            item_i = seq_codes[i]
            left = max(0, i - CF_WINDOW)
            right = min(n, i + CF_WINDOW + 1)

            for j in range(left, right):
                if i == j:
                    continue
                item_j = seq_codes[j]
                if item_i == item_j:
                    continue

                w = 1.0 / abs(i - j)
                co_counts[item_i][item_j] += w

    neighbors = {}
    for item_i, nbr_map in tqdm(co_counts.items(), desc="normalizing ItemCF"):
        arr = []
        cnt_i = item_cnt[item_i]
        for item_j, raw_score in nbr_map.items():
            cnt_j = item_cnt[item_j]
            if cnt_i <= 0 or cnt_j <= 0:
                continue
            sim = raw_score / math.sqrt(cnt_i * cnt_j)
            arr.append((int(item_j), float(sim)))

        arr.sort(key=lambda x: x[1], reverse=True)
        neighbors[int(item_i)] = arr[:CF_TOPN_PER_ITEM]

    global_hot_top = [(int(k), float(v)) for k, v in global_hot.most_common(HOT_ROUTE_TOPK * 5)]

    pid_hot_top = {}
    for pid, cnt in tqdm(pid_hot.items(), desc="building pid-hot lists"):
        pid_hot_top[int(pid)] = [(int(k), float(v)) for k, v in cnt.most_common(HOT_ROUTE_TOPK * 3)]

    return neighbors, global_hot_top, pid_hot_top

itemcf_neighbors, global_hot_top, pid_hot_top = build_itemcf_and_hot(user_event)

print("ItemCF items with neighbors:", len(itemcf_neighbors))
print("Global hot size:", len(global_hot_top))
print("PID hot size:", len(pid_hot_top))

# -----------------------
# ItemCF route
# -----------------------
def itemcf_route(example, topk=CF_ROUTE_TOPK):
    recent_items = [int(x) for x in example["hist_adg"][-CF_QUERY_RECENT:] if int(x) != 0]
    if len(recent_items) == 0:
        return []

    score_map = defaultdict(float)

    recent_items_rev = list(reversed(recent_items))
    for idx, item in enumerate(recent_items_rev):
        base_w = 1.0 / (1.0 + idx)
        nbrs = itemcf_neighbors.get(int(item), [])
        for nbr, sim in nbrs:
            score_map[int(nbr)] += base_w * float(sim)

    arr = list(score_map.items())
    arr.sort(key=lambda x: x[1], reverse=True)
    return arr[:topk]

# -----------------------
# Hot route
# -----------------------
def hot_route(example, topk=HOT_ROUTE_TOPK):
    pid = int(example["target_pid"])
    pid_list = pid_hot_top.get(pid, [])

    if len(pid_list) >= topk:
        return pid_list[:topk]

    seen = set()
    out = []

    for item, sc in pid_list:
        if item not in seen:
            seen.add(item)
            out.append((int(item), float(sc)))
        if len(out) >= topk:
            return out

    for item, sc in global_hot_top:
        if item not in seen:
            seen.add(item)
            out.append((int(item), float(sc)))
        if len(out) >= topk:
            break

    return out[:topk]

# -----------------------
# Precompute DSSM routes
# -----------------------
train_dt_routes = dual_tower_route(train_examples, topk=DT_ROUTE_TOPK, batch_size=256)
val_dt_routes   = dual_tower_route(val_examples,   topk=DT_ROUTE_TOPK, batch_size=256)
test_dt_routes  = dual_tower_route(test_examples,  topk=DT_ROUTE_TOPK, batch_size=256)

# -----------------------
# Build rerank dataset
# -----------------------
FEATURE_NAMES = [
    "from_dt",
    "dt_score",
    "dt_rank_inv",
    "from_cf",
    "cf_score",
    "cf_rank_inv",
    "from_hot",
    "hot_score",
    "hot_rank_inv",
    "route_count",
    "log_item_pop",
    "log_recent_pop",
    "same_cate_last",
    "same_brand_last",
    "recent_cate_match_frac",
    "recent_brand_match_frac",
    "price_norm",
    "abs_price_diff_last",
    "hist_len",
]

def build_rerank_matrix(examples, dt_routes, split_name):
    X_rows = []
    y_rows = []
    group = []

    dt_hit_50 = 0
    cf_hit_50 = 0
    hot_hit_50 = 0
    union_hit = 0

    for ex, dt_route_list in tqdm(zip(examples, dt_routes), total=len(examples), desc=f"building rerank matrix: {split_name}"):
        pos = int(ex["pos_adg"])

        dt_map = {}
        for rank, (item, score) in enumerate(dt_route_list, start=1):
            dt_map[int(item)] = (float(score), int(rank))

        cf_route_list = itemcf_route(ex, topk=CF_ROUTE_TOPK)
        cf_map = {}
        for rank, (item, score) in enumerate(cf_route_list, start=1):
            cf_map[int(item)] = (float(score), int(rank))

        hot_route_list = hot_route(ex, topk=HOT_ROUTE_TOPK)
        hot_map = {}
        for rank, (item, score) in enumerate(hot_route_list, start=1):
            hot_map[int(item)] = (float(score), int(rank))

        if pos in {item for item, _ in dt_route_list[:50]}:
            dt_hit_50 += 1
        if pos in {item for item, _ in cf_route_list[:50]}:
            cf_hit_50 += 1
        if pos in {item for item, _ in hot_route_list[:50]}:
            hot_hit_50 += 1

        pos_retrieved_before_force = (pos in dt_map) or (pos in cf_map) or (pos in hot_map)
        if pos_retrieved_before_force:
            union_hit += 1

        candidate_items = set(dt_map.keys()) | set(cf_map.keys()) | set(hot_map.keys())
        candidate_items.add(pos)  # force positive in rerank candidate set for supervised training

        last_cate = int(ex["last_cate"])
        last_brand = int(ex["last_brand"])
        last_price = float(ex["last_price"])

        hist_cate_recent = ex["hist_cate"][-5:] if len(ex["hist_cate"]) > 0 else np.array([], dtype=np.int64)
        hist_brand_recent = ex["hist_brand"][-5:] if len(ex["hist_brand"]) > 0 else np.array([], dtype=np.int64)

        rows_this_group = 0

        for item in candidate_items:
            item = int(item)

            dt_score, dt_rank = dt_map.get(item, (0.0, 0))
            cf_score, cf_rank = cf_map.get(item, (0.0, 0))
            hot_score, hot_rank = hot_map.get(item, (0.0, 0))

            from_dt = 1.0 if dt_rank > 0 else 0.0
            from_cf = 1.0 if cf_rank > 0 else 0.0
            from_hot = 1.0 if hot_rank > 0 else 0.0

            route_count = from_dt + from_cf + from_hot

            cate = int(cate_by_adg[item]) if item < len(cate_by_adg) else 0
            brand = int(brand_by_adg[item]) if item < len(brand_by_adg) else 0
            price = float(price_by_adg[item]) if item < len(price_by_adg) else 0.0

            same_cate_last = 1.0 if (cate != 0 and cate == last_cate) else 0.0
            same_brand_last = 1.0 if (brand != 0 and brand == last_brand) else 0.0

            if len(hist_cate_recent) > 0:
                recent_cate_match_frac = float(np.mean(hist_cate_recent == cate))
            else:
                recent_cate_match_frac = 0.0

            if len(hist_brand_recent) > 0:
                recent_brand_match_frac = float(np.mean(hist_brand_recent == brand))
            else:
                recent_brand_match_frac = 0.0

            row = [
                from_dt,
                float(dt_score),
                1.0 / (1.0 + dt_rank) if dt_rank > 0 else 0.0,

                from_cf,
                float(cf_score),
                1.0 / (1.0 + cf_rank) if cf_rank > 0 else 0.0,

                from_hot,
                float(hot_score),
                1.0 / (1.0 + hot_rank) if hot_rank > 0 else 0.0,

                float(route_count),
                float(np.log1p(total_clicks[item])) if item < len(total_clicks) else 0.0,
                float(np.log1p(recent_clicks[item])) if item < len(recent_clicks) else 0.0,
                same_cate_last,
                same_brand_last,
                float(recent_cate_match_frac),
                float(recent_brand_match_frac),
                float(price),
                float(abs(price - last_price)),
                float(ex["lens"]),
            ]

            X_rows.append(row)
            y_rows.append(1 if item == pos else 0)
            rows_this_group += 1

        group.append(rows_this_group)

    route_stats = {
        "dt_recall@50_on_sampled_queries": dt_hit_50 / max(len(examples), 1),
        "cf_recall@50_on_sampled_queries": cf_hit_50 / max(len(examples), 1),
        "hot_recall@50_on_sampled_queries": hot_hit_50 / max(len(examples), 1),
        "union_candidate_hit_rate_before_force": union_hit / max(len(examples), 1),
        "num_queries": len(examples),
        "num_rows": len(X_rows),
        "avg_candidates_per_query": len(X_rows) / max(len(examples), 1),
    }

    X = np.asarray(X_rows, dtype=np.float32)
    y = np.asarray(y_rows, dtype=np.int32)

    return X, y, group, route_stats

X_train, y_train, g_train, stats_train = build_rerank_matrix(train_examples, train_dt_routes, "train")
X_val,   y_val,   g_val,   stats_val   = build_rerank_matrix(val_examples,   val_dt_routes,   "val")
X_test,  y_test,  g_test,  stats_test  = build_rerank_matrix(test_examples,  test_dt_routes,  "test")

print("FEATURE_NAMES:", FEATURE_NAMES)
print("train stats:", stats_train)
print("val stats:", stats_val)
print("test stats:", stats_test)

# -----------------------
# Train XGBRanker
# -----------------------
ranker = xgb.XGBRanker(
    objective="rank:ndcg",
    eval_metric="ndcg@50",
    tree_method="hist",
    learning_rate=XGB_LR,
    max_depth=XGB_MAX_DEPTH,
    n_estimators=XGB_NUM_BOOST_ROUND,
    subsample=XGB_SUBSAMPLE,
    colsample_bytree=XGB_COLSAMPLE,
    random_state=SEED,
)

ranker.fit(
    X_train,
    y_train,
    group=g_train,
    eval_set=[(X_val, y_val)],
    eval_group=[g_val],
    verbose=True,
)

# -----------------------
# Grouped evaluation helpers
# -----------------------
def grouped_recall_at_k(pred, y, group, k):
    hits = 0
    idx = 0
    num_q = len(group)

    for g in group:
        p = pred[idx:idx+g]
        yy = y[idx:idx+g]

        kk = min(k, g)
        top_idx = np.argpartition(-p, kk - 1)[:kk]
        if np.any(yy[top_idx] > 0):
            hits += 1

        idx += g

    return hits / max(num_q, 1)

def evaluate_ranker(ranker, X, y, group, split_name):
    pred = ranker.predict(X)

    metrics = {
        "recall@10": grouped_recall_at_k(pred, y, group, 10),
        "recall@20": grouped_recall_at_k(pred, y, group, 20),
        "recall@50": grouped_recall_at_k(pred, y, group, 50),
        "recall@100": grouped_recall_at_k(pred, y, group, 100),
    }

    print(f"{split_name} rerank metrics:", metrics)
    return metrics

val_metrics = evaluate_ranker(ranker, X_val, y_val, g_val, "VAL")
test_metrics = evaluate_ranker(ranker, X_test, y_test, g_test, "TEST")

# -----------------------
# Feature importance
# -----------------------
importances = ranker.feature_importances_
feat_imp = sorted(zip(FEATURE_NAMES, importances), key=lambda x: x[1], reverse=True)
print("Top feature importances:")
for name, imp in feat_imp[:15]:
    print(f"{name:>28s} : {imp:.6f}")

DEVICE: cuda
{'train_queries_all': 588294, 'val_queries_all': 142596, 'test_queries_all': 142690}
{'train_queries_used': 3000, 'val_queries_used': 1000, 'test_queries_used': 3000}


building test query examples: 100%|██████████| 3000/3000 [00:00<00:00, 42835.74it/s]


active item matrix: (171828, 128)


building pid-hot lists: 100%|██████████| 2/2 [00:00<00:00, 269.97it/s]


ItemCF items with neighbors: 116065
Global hot size: 500
PID hot size: 2


building rerank matrix: test: 100%|██████████| 3000/3000 [00:14<00:00, 206.06it/s]


FEATURE_NAMES: ['from_dt', 'dt_score', 'dt_rank_inv', 'from_cf', 'cf_score', 'cf_rank_inv', 'from_hot', 'hot_score', 'hot_rank_inv', 'route_count', 'log_item_pop', 'log_recent_pop', 'same_cate_last', 'same_brand_last', 'recent_cate_match_frac', 'recent_brand_match_frac', 'price_norm', 'abs_price_diff_last', 'hist_len']
train stats: {'dt_recall@50_on_sampled_queries': 0.087, 'cf_recall@50_on_sampled_queries': 0.486, 'hot_recall@50_on_sampled_queries': 0.04, 'union_candidate_hit_rate_before_force': 0.6236666666666667, 'num_queries': 3000, 'num_rows': 1393233, 'avg_candidates_per_query': 464.411}
val stats: {'dt_recall@50_on_sampled_queries': 0.058, 'cf_recall@50_on_sampled_queries': 0.057, 'hot_recall@50_on_sampled_queries': 0.046, 'union_candidate_hit_rate_before_force': 0.255, 'num_queries': 1000, 'num_rows': 476448, 'avg_candidates_per_query': 476.448}
test stats: {'dt_recall@50_on_sampled_queries': 0.069, 'cf_recall@50_on_sampled_queries': 0.06566666666666666, 'hot_recall@50_on_sampl